# 01 - Tropical Cyclone Hazard

**Project:** Climate-Conditioned Tropical Cyclone Risk, Coastal Odisha Belt

**What this notebook establishes:** a stochastic tropical cyclone wind hazard set over the
coastal Odisha study domain, validated against two historical landfalling events.

**What it hands downstream:** a `TropCyclone` hazard object containing 2,754 synthetic
events evaluated at 525 spatial centroids, saved to disk for use in `03_impact.ipynb`.

---

### Approach

The hazard module answers one question: *how severe, and how often?* It does not yet
involve assets or money, those enter in notebooks 02 and 03.

The workflow is:

1. Load observed cyclone tracks for the North Indian Ocean basin (IBTrACS)
2. Screen to storms that could plausibly affect the study domain
3. Construct a regular spatial grid of evaluation points (centroids)
4. Expand the historical record into a stochastic event set, since ~50 years of
   observations cannot support return-period estimates
5. Compute a wind field for every event at every centroid
6. Validate the result against known storms before trusting it

Step 4 is the one that deserves emphasis. The observational record is far too short to
estimate a 1-in-100-year wind directly. Stochastic perturbation generates physically
plausible alternative tracks from each historical storm, expanding the sample without
inventing new climatology.

In [1]:
import sys
print(sys.executable)

/home/navneet/miniconda3/envs/climada_env/bin/python


In [2]:
import climada
from climada.hazard import TCTracks

print("CLIMADA imported successfully")
print(climada)

CLIMADA imported successfully
<module 'climada' from '/home/navneet/miniconda3/envs/climada_env/lib/python3.11/site-packages/climada/__init__.py'>


## Environment check

CLIMADA has a heavy geospatial dependency stack. This cell confirms the correct
environment is active before anything else runs.

# Odisha Cyclone Risk - Hazard Modelling

## 01. Tropical Cyclone Hazard

This notebook develops the tropical cyclone hazard component of the
Odisha cyclone risk model using CLIMADA and IBTrACS data.

Workflow:
1. Load North Indian Ocean cyclone tracks
2. Validate the track dataset visually
3. Identify cyclones relevant to the Odisha region
4. Generate spatial wind hazard
5. Validate and save the hazard dataset

In [3]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from climada.hazard import TCTracks, TropCyclone, Centroids

print("Imports successful")

Imports successful


## 1. North Indian Ocean Cyclone Tracks

We use IBTrACS tropical cyclone observations for the North Indian Ocean.
The dataset is first inspected visually before any spatial filtering or
hazard calculation is performed.

### Loading the observational record

IBTrACS (International Best Track Archive for Climate Stewardship) is the standard
consolidated record of global tropical cyclone tracks. The `usa` provider is used for
consistency of wind reporting convention across the record.

The North Indian Ocean (`NI`) basin covers both the Bay of Bengal and the Arabian Sea.
Only a subset of these storms will be relevant to Odisha, which the next step addresses.

In [4]:
tracks_ni = TCTracks.from_ibtracs_netcdf(
    provider="usa",
    basin="NI"
)

print(f"Usable cyclone events: {len(tracks_ni.data)}")

2026-09-04 11:06:34,043 - climada.hazard.tc_tracks - WARNING - 1703 storm events are discarded because no valid wind/pressure values have been found: 1842298N11080, 1845336N10074, 1854303N14072, 1877135N10083, 1877192N21090, ...
Usable cyclone events: 146


## 2. Track Sanity Check

Before proceeding to hazard modelling, the cyclone tracks are plotted
to verify that the dataset has the expected spatial distribution over
the North Indian Ocean, including the Bay of Bengal.

In [5]:
tracks_ni.plot()

plt.savefig(
    "../outputs/north_indian_ocean_tracks.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()

## 3. Identify Odisha-Relevant Candidate Storms

A broad spatial screening is applied to identify cyclone tracks
passing through the region surrounding Odisha. This is a candidate
selection step; final relevance is determined from the resulting
spatial wind hazard rather than from track location alone.

### Screening to the study region

A deliberately **broad** box (15–23°N, 80–90°E) is used here, wider than the eventual
hazard grid. The reasoning: a storm passing just outside the modelling domain can still
generate damaging winds inside it, so screening too tightly at this stage would discard
relevant events.

Final relevance is determined by the computed wind field, not by track position - a storm
that enters the box but produces no meaningful wind over the domain simply contributes a
zero row to the intensity matrix.

In [6]:
odisha_candidates = [
    i for i, track in enumerate(tracks_ni.data)
    if (
        (track.lat.values >= 15) &
        (track.lat.values <= 23) &
        (track.lon.values >= 80) &
        (track.lon.values <= 90)
    ).any()
]

print(f"Candidate storms: {len(odisha_candidates)}")

Candidate storms: 54


## 4. Odisha Hazard Domain

A regular spatial grid is created around Odisha and the adjacent Bay of
Bengal. These centroids are the locations at which CLIMADA will calculate
cyclone wind intensity.

### Constructing the hazard grid

The modelling domain is defined as **17–22°N, 82–88°E at 0.25° resolution**, giving
21 × 25 = 525 centroids (≈27 km spacing).

**Why the coastal belt rather than the full state of Odisha.** Tropical cyclone wind
intensity decays rapidly after landfall, so inland districts contribute little wind hazard
while adding computation and diluting spatial loss metrics. The domain is scoped to where
wind hazard is materially concentrated, and is named the *coastal Odisha belt* throughout
the project rather than "Odisha", to avoid implying full state coverage.

In [7]:
lat = np.arange(17, 22.01, 0.25)
lon = np.arange(82, 88.01, 0.25)

lon_grid, lat_grid = np.meshgrid(lon, lat)

centroids_odisha = Centroids(
    lat=lat_grid.ravel(),
    lon=lon_grid.ravel()
)

print(f"Number of centroids: {len(centroids_odisha.lat)}")

Number of centroids: 525


## 5. Generate Tropical Cyclone Wind Hazard

The candidate cyclone tracks are converted into spatial wind-intensity
fields over the Odisha analysis domain using CLIMADA's tropical cyclone
wind-field model.

In [8]:
tracks_odisha = TCTracks(
    data=[tracks_ni.data[i] for i in odisha_candidates]
)

print(f"Odisha candidate events: {len(tracks_odisha.data)}")

Odisha candidate events: 54


### First-pass wind field

An initial hazard computation on the historical tracks only, with the distance-to-coast
requirement bypassed. This is a smoke test that the track data, centroids, and wind model
are wired together correctly. The production hazard set is built later in this notebook,
after the distance-to-coast issue is resolved properly.

In [9]:
hazard_odisha = TropCyclone.from_tracks(
    tracks_odisha,
    centroids_odisha,
    ignore_distance_to_coast=True
)

In [10]:
print(hazard_odisha.intensity.shape)

(54, 525)


## Stochastic event set generation

54 historical storms is not enough to estimate return periods. With roughly one relevant
storm every year or two, the observational record supports meaningful statistics only out
to a few decades far short of the 1-in-100 or 1-in-200 estimates the analysis requires.

CLIMADA's `calc_perturbed_trajectories` addresses this by generating synthetic variations
of each historical track, perturbing translation direction and speed while preserving the
statistical properties of the underlying climatology. This is the standard approach to
building a stochastic catalogue, and is what commercial catastrophe platforms do internally
at greater sophistication.

In [11]:
print(hasattr(tracks_ni, "calc_perturbed_trajectories"))
print(hasattr(tracks_odisha, "calc_perturbed_trajectories"))

True
True


### Preserving the historical set separately

Two copies are kept deliberately:

- `tracks_hist` - the unmodified historical tracks, retained for validation against
  observed events later in this notebook
- `tracks_stoch` - the working copy that will be perturbed

Keeping the historical record untouched matters, because the validation step needs to
compare modelled output against *observations*, not against perturbed derivatives of them.

In [12]:
import copy

tracks_hist = copy.deepcopy(tracks_odisha)

In [13]:
tracks_stoch = copy.deepcopy(tracks_odisha)

### Temporal standardisation

Tracks are resampled to a uniform 1-hour timestep before perturbation. IBTrACS reports at
mixed intervals (typically 3- or 6-hourly, with higher frequency near landfall), and the
perturbation algorithm assumes regular spacing. Skipping this step produces perturbed tracks
whose geometry is distorted by the underlying reporting irregularity.

The following cell confirms that all tracks now share a single timestep value.

In [14]:
tracks_stoch.equal_timestep(time_step_h=1)

In [15]:
print(
    sorted(
        set(
            np.concatenate([
                track.time_step.values
                for track in tracks_stoch.data
            ])
        )
    )
)

[np.float64(1.0)]


### Perturbation - first pass (9 members)

An initial expansion with 9 synthetic tracks per historical storm, used to verify the
mechanism works and to inspect the resulting event naming and attributes.

`decay=True` applies post-landfall intensity decay, which matters here because the study
domain includes land. A fixed `seed` makes the catalogue reproducible.

> **Note:** this 9-member catalogue is superseded later in the notebook by a 50-member
> expansion, which is the version used for all downstream loss analysis.

In [16]:
tracks_stoch.calc_perturbed_trajectories(
    nb_synth_tracks=9,
    seed=54,
    decay=True,
    use_global_decay_params=True
)

In [17]:
print("Historical tracks:", len(tracks_odisha.data))
print("Stochastic tracks:", len(tracks_stoch.data))

Historical tracks: 54
Stochastic tracks: 540


In [18]:
print(tracks_stoch.data[0].attrs)

{'max_sustained_wind_unit': 'kn', 'central_pressure_unit': 'mb', 'orig_event_flag': True, 'data_provider': 'ibtracs_usa', 'category': np.int64(-1), 'name': 'UNNAMED', 'sid': '2001313N12084', 'id_no': 2001313012084.0}


In [19]:
print(tracks_stoch.data[-1].attrs)

{'max_sustained_wind_unit': 'kn', 'central_pressure_unit': 'mb', 'orig_event_flag': False, 'data_provider': 'ibtracs_usa', 'category': np.int64(0), 'name': 'MONTHA_gen9', 'sid': '2025298N11089_gen9', 'id_no': 2025298011089.09}


## Distance to coast - resolving an external dependency failure

CLIMADA's tropical cyclone wind model requires a distance-to-coast value at each centroid,
which it obtains by downloading a pre-computed NASA raster. That download returned
**HTTP 403 Forbidden**, blocking the pipeline.

The cell below prints the expected file location and source URL, confirming the dependency
CLIMADA is trying to satisfy.

Rather than wait on an external service, distance-to-coast is computed directly from
Natural Earth coastline geometry using WGS84 geodesic distances. This removes the
dependency entirely and makes the pipeline reproducible without the NASA download.

In [20]:
from climada.util.constants import SYSTEM_DIR
from climada.util.config import CONFIG

print("SYSTEM_DIR:", SYSTEM_DIR)
print("URL:", CONFIG.util.coordinates.dist_to_coast_nasa_url.str())
print("Expected TIF:", CONFIG.util.coordinates.dist_to_coast_nasa_tif.str())

SYSTEM_DIR: /home/navneet/climada/data
URL: https://oceancolor.gsfc.nasa.gov/docs/distfromcoast/GMT_intermediate_coast_distance_01d.zip
Expected TIF: GMT_intermediate_coast_distance_01d.tif


In [21]:
print(centroids_odisha.gdf.head())
print(centroids_odisha.gdf.crs)

           geometry region_id on_land
0     POINT (82 17)      None    None
1  POINT (82.25 17)      None    None
2   POINT (82.5 17)      None    None
3  POINT (82.75 17)      None    None
4     POINT (83 17)      None    None
EPSG:4326


In [22]:
import geopandas as gpd
import cartopy.feature as cfeature

### Coastline reference geometry

Natural Earth 10 m physical coastlines are used as the reference geometry. This is a
well-established open dataset at a resolution appropriate to a 0.25° (≈27 km) grid.

In [23]:
import cartopy.io.shapereader as shpreader

shpfilename = shpreader.natural_earth(
    resolution="10m",
    category="physical",
    name="coastline"
)

coastline = gpd.read_file(shpfilename)

print(coastline.crs)
print(len(coastline))

EPSG:4326
4133


In [24]:
from pyproj import Geod

geod = Geod(ellps="WGS84")

### Geodesic distance calculation

For each centroid, the function finds the nearest point on the nearest coastline geometry,
then computes the true **WGS84 geodesic distance** between the two.

Geodesic distance is used rather than a projected planar approximation because it is exact
on the ellipsoid and requires no justification about projection distortion across the
domain. At 525 points the computational cost is negligible.

In [25]:
from shapely.ops import nearest_points
from pyproj import Geod

geod = Geod(ellps="WGS84")

def distance_to_coast(point, coastline):
    # Find the nearest coastline geometry
    distances = coastline.geometry.distance(point)
    nearest_idx = distances.idxmin()

    # Find the nearest point on that coastline geometry
    coast_point = nearest_points(
        point,
        coastline.geometry.loc[nearest_idx]
    )[1]

    # Calculate true WGS84 geodesic distance in metres
    _, _, distance_m = geod.inv(
        point.x,
        point.y,
        coast_point.x,
        coast_point.y
    )

    return distance_m

In [26]:
centroids_odisha.gdf["dist_coast"] = (
    centroids_odisha.gdf.geometry.apply(
        lambda point: distance_to_coast(point, coastline)
    )
)

In [27]:
print(centroids_odisha.gdf[["geometry", "dist_coast"]].head())

           geometry    dist_coast
0     POINT (82 17)  28031.418297
1  POINT (82.25 17)   3986.989192
2   POINT (82.5 17)  16393.227282
3  POINT (82.75 17)  34715.546319
4     POINT (83 17)  46611.521210


In [28]:
print(
    "Minimum distance:",
    centroids_odisha.gdf["dist_coast"].min(),
    "m"
)

print(
    "Maximum distance:",
    centroids_odisha.gdf["dist_coast"].max(),
    "m"
)

Minimum distance: 177.8751038822295 m
Maximum distance: 418507.0132777567 m


In [29]:
print("dist_coast" in centroids_odisha.gdf.columns)

True


### Production wind field - 9-member catalogue

With distance-to-coast now populated, the wind field is recomputed without bypassing the
requirement. The resulting sparse matrix has dimensions *(events × centroids)*.

In [30]:
hazard_odisha = TropCyclone.from_tracks(
    tracks_stoch,
    centroids_odisha
)

In [31]:
print(hazard_odisha.intensity.shape)
print(len(hazard_odisha.event_name))

(540, 525)
540


## Hazard validation

Before the hazard set is used for anything, it is checked against physical expectations and
against observations. Four checks are applied:

1. **Magnitude** - is the maximum modelled wind physically plausible for this basin?
2. **Sparsity** - do most storms correctly miss most locations?
3. **Structure** - does an individual event footprint look like a cyclone?
4. **Historical agreement** - does the model reproduce known storms?

A hazard set that fails any of these should not be carried forward, regardless of how
cleanly the code ran.

In [32]:
event_max = hazard_odisha.intensity.max(axis=1).toarray().ravel()

print("Maximum wind:", event_max.max(), "m/s")

Maximum wind: 69.24113223854867 m/s


In [33]:
strongest = np.argsort(event_max)[-10:][::-1]

for i in strongest:
    print(
        i,
        hazard_odisha.event_name[i],
        round(event_max[i], 2),
        "m/s"
    )

359 2019116N02090_gen9 69.24 m/s
357 2019116N02090_gen7 69.14 m/s
355 2019116N02090_gen5 68.89 m/s
358 2019116N02090_gen8 68.82 m/s
356 2019116N02090_gen6 68.51 m/s
352 2019116N02090_gen2 68.35 m/s
350 2019116N02090 68.05 m/s
354 2019116N02090_gen4 67.82 m/s
351 2019116N02090_gen1 67.62 m/s
353 2019116N02090_gen3 67.45 m/s


### Sparsity check

Most cyclones affect only a small portion of the domain, so the intensity matrix should be
overwhelmingly zero. A dense matrix would indicate that wind is being spread unphysically
across the grid.

In [34]:
intensity = hazard_odisha.intensity

total = intensity.shape[0] * intensity.shape[1]
zeros = intensity.nnz

print("Total event-centroid pairs:", total)
print("Non-zero pairs:", zeros)
print("Non-zero percentage:", 100 * zeros / total)
print("Zero percentage:", 100 * (1 - zeros / total))

Total event-centroid pairs: 283500
Non-zero pairs: 20963
Non-zero percentage: 7.394356261022928
Zero percentage: 92.60564373897708


Hazard sanity check : Fani 2019:
The original Fani track has a reported maximum sustained wind of 150 kt at approximately 17.6°N, 84.8°E. The CLIMADA-derived wind footprint produces a maximum of 68.05 m/s (≈132 kt) at 17.75°N, 85.0°E, approximately 27 km from the IBTrACS maximum location. The modeled footprint therefore reproduces the spatial location and order of magnitude of the historical maximum reasonably well, while the difference in wind magnitude is retained as model uncertainty rather than treated as an exact match.

### Event frequency

Each event in the catalogue carries an annual frequency. Because perturbation divides each
historical storm's frequency among its synthetic members, the **total** frequency across the
catalogue should be preserved by the expansion.

This is worth verifying explicitly: an expansion that inflated total frequency would silently
scale every downstream loss metric.

In [35]:
print("Number of events:", len(hazard_odisha.frequency))
print("Unique frequencies:", np.unique(hazard_odisha.frequency))

Number of events: 540
Unique frequencies: [0.004]


In [36]:
print(
    "Total annual event frequency:",
    hazard_odisha.frequency.sum()
)

Total annual event frequency: 2.16


## Production catalogue - 50-member expansion

The 9-member catalogue is now superseded. Nine synthetic tracks per historical storm gives
486 events, which remains thin in the tail. Expanding to **50 members per storm** produces
2,754 events and substantially improves the stability of return-period estimates.

The same seed and decay settings are used, so the expansion differs only in size.

**This 50-member catalogue (`hazard_odisha_50`) is the version used for all downstream loss
analysis.**

In [37]:
import copy

tracks_stoch_50 = copy.deepcopy(tracks_odisha)

tracks_stoch_50.equal_timestep(time_step_h=1)

In [38]:
print(
    sorted(
        set(
            np.concatenate([
                track.time_step.values
                for track in tracks_stoch_50.data
            ])
        )
    )
)

[np.float64(1.0)]


In [39]:
tracks_stoch_50.calc_perturbed_trajectories(
    nb_synth_tracks=50,
    seed=54,
    decay=True,
    use_global_decay_params=True
)

In [40]:
print("Historical tracks:", len(tracks_odisha.data))
print("50-member stochastic tracks:", len(tracks_stoch_50.data))

Historical tracks: 54
50-member stochastic tracks: 2754


In [41]:
hazard_odisha_50 = TropCyclone.from_tracks(
    tracks_stoch_50,
    centroids_odisha
)

In [42]:
print(hazard_odisha_50.intensity.shape)
print(len(hazard_odisha_50.event_name))
print(hazard_odisha_50.frequency.sum())

(2754, 525)
2754
2.1599999999999993


In [61]:
intensity = hazard_odisha_50.intensity

total = intensity.shape[0] * intensity.shape[1]
zeros = intensity.nnz

print("Total event-centroid pairs:", total)
print("Non-zero pairs:", zeros)
print("Non-zero percentage:", 100 * zeros / total)
print("Zero percentage:", 100 * (1 - zeros / total))

Total event-centroid pairs: 1445850
Non-zero pairs: 102558
Non-zero percentage: 7.093266936404191
Zero percentage: 92.90673306359581


Hazard stage conclusion:
Historical IBTrACS tropical cyclone tracks were screened to identify 54 events entering the Odisha study domain. Tracks were standardized to a 1-hour temporal resolution and expanded using CLIMADA's stochastic track perturbation methodology, producing 2,754 events from 50 perturbations per historical track. Wind footprints were calculated over 525 spatial centroids. A local WGS84 geodesic distance-to-coast calculation was used because CLIMADA's NASA distance-to-coast endpoint was inaccessible. Initial validation using Fani (2019) showed a spatially coherent footprint and a modeled maximum of approximately 68 m/s, compared with an IBTrACS maximum of 150 kt. The stochastic catalogue represents plausible perturbations of historical events and should not be interpreted as 2,754 independent historical observations.

### Historical validation: Cyclone Fani (2019)

Fani made landfall near Puri in May 2019 as one of the strongest storms to strike Odisha in
decades, making it a natural benchmark. Two things are checked:

- **Spatial agreement** - does the modelled maximum occur near the observed maximum?
- **Magnitude agreement** - is the modelled peak wind of the right order?

These are treated separately and deliberately, because they carry different weight. Spatial
agreement validates that track processing, wind-field generation, and the centroid grid are
correctly aligned. Magnitude agreement is weaker evidence, since modelled and reported winds
may use different averaging conventions.

In [44]:
# ============================================================
# FANI VALIDATION
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. Find Fani in historical IBTrACS tracks
# ------------------------------------------------------------

fani_event = "2019116N02090"

fani_indices = [
    i for i, track in enumerate(tracks_hist.data)
    if track.attrs.get("sid") == fani_event
]

if len(fani_indices) == 0:
    raise ValueError(f"{fani_event} not found in tracks_hist")

fani_idx = fani_indices[0]
fani = tracks_hist.data[fani_idx]

print("Fani historical track")
print("Name:", fani.attrs.get("name"))
print("SID:", fani.attrs.get("sid"))

# Historical maximum wind
wind = np.asarray(fani.max_sustained_wind.values)
max_track_idx = np.nanargmax(wind)

print(
    "IBTrACS maximum wind:",
    wind[max_track_idx],
    fani.max_sustained_wind_unit
)

print(
    "Time:",
    fani.time.values[max_track_idx]
)

print(
    "Position:",
    fani.lat.values[max_track_idx],
    fani.lon.values[max_track_idx]
)

# ------------------------------------------------------------
# 2. Find Fani in final hazard catalogue
# ------------------------------------------------------------

event_names = np.asarray(
    hazard_odisha_50.event_name
).astype(str)

fani_matches = np.where(
    event_names == fani_event
)[0]

if len(fani_matches) == 0:
    raise ValueError(
        f"{fani_event} not found in hazard_odisha_50"
    )

fani_hazard_idx = fani_matches[0]

print(
    "\nFani hazard index:",
    fani_hazard_idx
)

# ------------------------------------------------------------
# 3. Modelled maximum Fani wind across the grid
# ------------------------------------------------------------

fani_intensity = hazard_odisha_50.intensity[
    fani_hazard_idx
]

if hasattr(fani_intensity, "toarray"):
    fani_intensity = (
        fani_intensity.toarray().ravel()
    )
else:
    fani_intensity = np.asarray(
        fani_intensity
    ).ravel()

fani_max_idx = np.argmax(fani_intensity)

print(
    "Modelled Fani maximum:",
    f"{fani_intensity[fani_max_idx]:.2f}",
    "m/s"
)

print(
    "Modelled Fani maximum:",
    f"{fani_intensity[fani_max_idx] * 3.6:.2f}",
    "km/h"
)

print(
    "Modelled Fani maximum location:",
    centroids_odisha.gdf.geometry.iloc[
        fani_max_idx
    ]
)

# ------------------------------------------------------------
# 4. Find centroid nearest to Fani's Odisha landfall
# ------------------------------------------------------------

fani_landfall_lat = 20.2
fani_landfall_lon = 85.9

dist2 = (
    (centroids_odisha.gdf.geometry.y - fani_landfall_lat) ** 2
    + (centroids_odisha.gdf.geometry.x - fani_landfall_lon) ** 2
)

nearest_idx = dist2.idxmin()

print(
    "Nearest landfall centroid:",
    nearest_idx
)

print(
    "Nearest centroid:",
    centroids_odisha.gdf.geometry.iloc[
        nearest_idx
    ]
)

print(
    "Modelled Fani wind at nearest landfall centroid:",
    f"{fani_intensity[nearest_idx]:.2f}",
    "m/s"
)

Fani historical track
Name: FANI
SID: 2019116N02090
IBTrACS maximum wind: 150.0 kn
Time: 2019-05-02T12:00:00.000000000
Position: 17.6 84.8

Fani hazard index: 1785
Modelled Fani maximum: 68.05 m/s
Modelled Fani maximum: 244.98 km/h
Modelled Fani maximum location: POINT (85 17.75)
Nearest landfall centroid: 341
Nearest centroid: POINT (86 20.25)
Modelled Fani wind at nearest landfall centroid: 50.21 m/s


### Return-period winds at Puri

A hazard-level return period check, independent of any loss calculation. Puri is chosen as a
reference point because it is a coastal location with well-documented cyclone exposure.

The calculation sorts events by intensity at that point, accumulates their frequencies, and
inverts to a return period. This gives wind speeds at the 1-in-10, 1-in-50 and 1-in-100
levels, which can be sanity-checked against regional cyclone climatology before any exposure
or vulnerability assumptions enter the analysis.

In [45]:
puri_lat = 19.81
puri_lon = 85.83

In [46]:
puri_dist = (
    (centroids_odisha.gdf.geometry.y - puri_lat)**2
    + (centroids_odisha.gdf.geometry.x - puri_lon)**2
)

puri_idx = puri_dist.idxmin()

print("Puri centroid index:", puri_idx)
print("Puri centroid:", centroids_odisha.gdf.geometry.iloc[puri_idx])

Puri centroid index: 290
Puri centroid: POINT (85.75 19.75)


In [47]:
puri_wind = hazard_odisha_50.intensity[:, puri_idx].toarray().ravel()

print("Maximum Puri wind:", puri_wind.max(), "m/s")
print("Maximum Puri wind:", puri_wind.max() * 3.6, "km/h")

Maximum Puri wind: 66.97005020921569 m/s
Maximum Puri wind: 241.09218075317648 km/h


In [48]:
puri_freq = hazard_odisha_50.frequency
puri_wind = hazard_odisha_50.intensity[:, puri_idx].toarray().ravel()

# Sort events by wind intensity
order = np.argsort(puri_wind)[::-1]

sorted_wind = puri_wind[order]
sorted_freq = puri_freq[order]

# Cumulative annual exceedance frequency
cum_freq = np.cumsum(sorted_freq)

# Return period = 1 / exceedance frequency
return_period = 1 / cum_freq

for rp in [10, 50, 100]:
    idx = np.where(return_period >= rp)[0][-1]
    print(
        f"1-in-{rp} year wind:",
        round(sorted_wind[idx], 2),
        "m/s |",
        round(sorted_wind[idx] * 3.6, 1),
        "km/h"
    )

1-in-10 year wind: 19.65 m/s | 70.7 km/h
1-in-50 year wind: 43.03 m/s | 154.9 km/h
1-in-100 year wind: 50.68 m/s | 182.5 km/h


### Second historical validation: Cyclone Phailin (2013)

A single validation event is one data point. Phailin, which made landfall near Gopalpur in
October 2013, provides an independent second check on the same pipeline.

In [49]:
# ============================================================
# PHAILIN VALIDATION
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. Find Phailin in historical IBTrACS tracks
# ------------------------------------------------------------

phailin_name = "PHAILIN"

phailin_indices = [
    i for i, track in enumerate(tracks_hist.data)
    if track.attrs.get("name") == phailin_name
]

if len(phailin_indices) == 0:
    raise ValueError(
        "PHAILIN not found in tracks_hist"
    )

phailin_idx = phailin_indices[0]
phailin = tracks_hist.data[phailin_idx]

print("Phailin historical track")
print("Name:", phailin.attrs.get("name"))
print("SID:", phailin.attrs.get("sid"))

# ------------------------------------------------------------
# 2. Historical IBTrACS maximum wind
# ------------------------------------------------------------

phailin_wind = np.asarray(
    phailin.max_sustained_wind.values
)

phailin_track_max_idx = np.nanargmax(
    phailin_wind
)

print(
    "IBTrACS maximum wind:",
    phailin_wind[phailin_track_max_idx],
    phailin.max_sustained_wind_unit
)

print(
    "Time:",
    phailin.time.values[phailin_track_max_idx]
)

print(
    "Position:",
    phailin.lat.values[phailin_track_max_idx],
    phailin.lon.values[phailin_track_max_idx]
)

# ------------------------------------------------------------
# 3. Find original Phailin event in FINAL hazard catalogue
# ------------------------------------------------------------

phailin_sid = phailin.attrs.get("sid")

if phailin_sid is None:
    raise ValueError(
        "Phailin track does not contain an SID"
    )

event_names = np.asarray(
    hazard_odisha_50.event_name
).astype(str)

phailin_matches = np.where(
    event_names == phailin_sid
)[0]

if len(phailin_matches) == 0:
    raise ValueError(
        f"{phailin_sid} not found in hazard_odisha_50"
    )

phailin_hazard_idx = phailin_matches[0]

print(
    "\nPhailin hazard index:",
    phailin_hazard_idx
)

print(
    "Selected event:",
    hazard_odisha_50.event_name[
        phailin_hazard_idx
    ]
)

# ------------------------------------------------------------
# 4. Modelled Phailin maximum wind
# ------------------------------------------------------------

phailin_intensity = hazard_odisha_50.intensity[
    phailin_hazard_idx
]

if hasattr(phailin_intensity, "toarray"):
    phailin_intensity = (
        phailin_intensity.toarray().ravel()
    )
else:
    phailin_intensity = np.asarray(
        phailin_intensity
    ).ravel()

phailin_model_max_idx = np.argmax(
    phailin_intensity
)

print(
    "Modelled Phailin maximum:",
    f"{phailin_intensity[phailin_model_max_idx]:.2f}",
    "m/s"
)

print(
    "Modelled Phailin maximum:",
    f"{phailin_intensity[phailin_model_max_idx] * 3.6:.2f}",
    "km/h"
)

print(
    "Modelled Phailin maximum location:",
    centroids_odisha.gdf.geometry.iloc[
        phailin_model_max_idx
    ]
)

Phailin historical track
Name: PHAILIN
SID: 2013281N12098
IBTrACS maximum wind: 140.0 kn
Time: 2013-10-11T12:00:00.000000000
Position: 16.7 87.7

Phailin hazard index: 1224
Selected event: 2013281N12098
Modelled Phailin maximum: 64.25 m/s
Modelled Phailin maximum: 231.31 km/h
Modelled Phailin maximum location: POINT (86.75 17.25)


The modeled wind footprints reproduce the approximate magnitude and spatial location of the historical maximum winds for both Fani (2019) and Phailin (2013), with modeled maxima approximately 11–12% below the corresponding IBTrACS track maxima.

## Tail stability test

Return-period estimates far into the tail are supported by only a handful of catalogue
events, so they may reflect sampling noise rather than climatology.

This is tested directly: the catalogue is split into two random halves and the 1-in-100-year
wind is estimated independently from each. Close agreement would indicate a stable tail.
Substantial divergence indicates the estimate remains sampling-sensitive and should be
reported with that caveat rather than as a precise figure.

In [50]:
rng = np.random.default_rng(123)

n_events = len(hazard_odisha_50.event_name)

indices = np.arange(n_events)
rng.shuffle(indices)

half_a = indices[:n_events // 2]
half_b = indices[n_events // 2:]

print("Half A:", len(half_a))
print("Half B:", len(half_b))

Half A: 1377
Half B: 1377


In [51]:
def return_period_wind(event_indices, return_period, hazard, location_idx):
    wind = hazard.intensity[event_indices, location_idx].toarray().ravel()
    freq = hazard.frequency[event_indices]

    order = np.argsort(wind)[::-1]

    sorted_wind = wind[order]
    sorted_freq = freq[order]

    cum_freq = np.cumsum(sorted_freq)
    rp = 1 / cum_freq

    valid = np.where(rp >= return_period)[0]

    if len(valid) == 0:
        return np.nan

    return sorted_wind[valid[-1]]

In [52]:
puri_100_a = return_period_wind(
    half_a, 100, hazard_odisha_50, puri_idx
)

puri_100_b = return_period_wind(
    half_b, 100, hazard_odisha_50, puri_idx
)

print("Half A 1-in-100 wind:", puri_100_a, "m/s")
print("Half B 1-in-100 wind:", puri_100_b, "m/s")

Half A 1-in-100 wind: 41.96635503644057 m/s
Half B 1-in-100 wind: 47.13708063291371 m/s


Tail stability: The 1-in-100-year wind estimate at the Puri grid point was 50.68 m/s for the full stochastic catalogue. When the catalogue was divided into two random halves, corresponding estimates were 41.97 and 47.14 m/s. The difference indicates that the extreme tail remains sensitive to stochastic sampling and should therefore not be interpreted as a precise 1-in-100 climatological estimate.

## Persisting the hazard set

The 50-member hazard object is serialised so that `03_impact.ipynb` can load it without
recomputing the wind fields, which is the most expensive step in the pipeline.

In [53]:
from pathlib import Path
import pickle

output_dir = Path("../outputs")

output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir.resolve())

Output directory: /home/navneet/Projects/odisha-cyclone-risk/outputs


In [54]:
with open(output_dir / "hazard_odisha_50.pkl", "wb") as f:
    pickle.dump(hazard_odisha_50, f)

print("Hazard saved.")
print("Events:", len(hazard_odisha_50.event_name))
print("Centroids:", len(hazard_odisha_50.centroids.gdf))

Hazard saved.
Events: 2754
Centroids: 525


In [55]:
print("Number of stochastic tracks:", len(tracks_stoch.data))
print("Number of hazard events:", len(hazard_odisha.event_name))

print("\nFirst 10 hazard event names:")
print(hazard_odisha.event_name[:10])

Number of stochastic tracks: 540
Number of hazard events: 540

First 10 hazard event names:
['2001313N12084', '2001313N12084_gen1', '2001313N12084_gen2', '2001313N12084_gen3', '2001313N12084_gen4', '2001313N12084_gen5', '2001313N12084_gen6', '2001313N12084_gen7', '2001313N12084_gen8', '2001313N12084_gen9']


## Summary figure

Generates the hazard overview figure used in the project README: historical candidate tracks,
the highest-loss stochastic event highlighted, and the modelled wind footprint across the
grid.

In [56]:
import os
import numpy as np
import matplotlib.pyplot as plt

figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Event identified from the completed loss analysis
# -------------------------------------------------------

strongest_event = "2019116N02090_gen4"

# Find event in the FINAL 2,754-event hazard catalogue
event_names = np.asarray(
    hazard_odisha_50.event_name
).astype(str)

matches = np.where(
    event_names == strongest_event
)[0]

if len(matches) == 0:
    raise ValueError(
        f"{strongest_event} not found in hazard_odisha_50"
    )

strongest_idx = matches[0]

print("Strongest-loss event:", strongest_event)
print("Hazard event index:", strongest_idx)

# -------------------------------------------------------
# Same ordering: tracks_stoch_50 ↔ hazard_odisha_50
# -------------------------------------------------------

strong_track = tracks_stoch_50.data[strongest_idx]

print(
    "Track name:",
    getattr(strong_track, "name", "UNNAMED")
)

# -------------------------------------------------------
# Wind intensity footprint
# -------------------------------------------------------

intensity = hazard_odisha_50.intensity[strongest_idx]

if hasattr(intensity, "toarray"):
    intensity = intensity.toarray().ravel()
else:
    intensity = np.asarray(intensity).ravel()

lon = np.asarray(
    hazard_odisha_50.centroids.lon
)

lat = np.asarray(
    hazard_odisha_50.centroids.lat
)

mask = intensity > 0

# -------------------------------------------------------
# Plot
# -------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 8))

# Historical candidate tracks
for track in tracks_odisha.data:
    ax.plot(
        np.asarray(track.lon),
        np.asarray(track.lat),
        linewidth=0.9,
        alpha=0.25
    )

# Highlight the actual highest-loss event
ax.plot(
    np.asarray(strong_track.lon),
    np.asarray(strong_track.lat),
    linewidth=2.8,
    label="Highest-loss event (₹70.97B)"
)

# Wind footprint
sc = ax.scatter(
    lon[mask],
    lat[mask],
    c=intensity[mask],
    s=35,
    alpha=0.75
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Maximum wind speed (m/s)")

ax.set_xlim(82, 88)
ax.set_ylim(17, 22)

ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_title(
    "Tropical Cyclone Hazard - Coastal Odisha"
)

ax.grid(alpha=0.25)
ax.legend(loc="upper left")

plt.tight_layout()

plt.savefig(
    figures_dir / "hazard_tracks_and_footprint.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Strongest-loss event: 2019116N02090_gen4
Hazard event index: 1789
Track name: FANI_gen4


In [57]:
print(len(tracks_odisha.data))

54


### Wind unit confirmation

Confirms the units of the IBTrACS `max_sustained_wind` field before any comparison against
modelled output in m/s. Comparing across unit conventions without checking is a common source
of apparent model error.

In [58]:
import numpy as np

# Find Fani in the original IBTrACS tracks
fani = None

for track in tracks_odisha.data:
    if track.sid == "2019116N02090":
        fani = track
        break

print("Storm:", fani.sid)

# Maximum IBTrACS wind
max_idx = np.nanargmax(fani.max_sustained_wind)

print("Maximum wind:", fani.max_sustained_wind[max_idx], "m/s")
print("Latitude:", fani.lat[max_idx])
print("Longitude:", fani.lon[max_idx])
print("Time:", fani.time[max_idx])

Storm: 2019116N02090
Maximum wind: <xarray.DataArray 'max_sustained_wind' ()> Size: 4B
array(150., dtype=float32)
Coordinates:
    time     datetime64[ns] 8B 2019-05-02T12:00:00
    lat      float32 4B 17.6
    lon      float32 4B 84.8 m/s
Latitude: <xarray.DataArray 'lat' ()> Size: 4B
array(17.6, dtype=float32)
Coordinates:
    time     datetime64[ns] 8B 2019-05-02T12:00:00
    lat      float32 4B 17.6
    lon      float32 4B 84.8
Longitude: <xarray.DataArray 'lon' ()> Size: 4B
array(84.8, dtype=float32)
Coordinates:
    time     datetime64[ns] 8B 2019-05-02T12:00:00
    lat      float32 4B 17.6
    lon      float32 4B 84.8
Time: <xarray.DataArray 'time' ()> Size: 8B
array('2019-05-02T12:00:00.000000000', dtype='datetime64[ns]')
Coordinates:
    time     datetime64[ns] 8B 2019-05-02T12:00:00
    lat      float32 4B 17.6
    lon      float32 4B 84.8


In [59]:
print("Wind unit:", fani.max_sustained_wind_unit)

Wind unit: kn


In [60]:
import numpy as np

model_max = hazard_odisha.intensity.max()

print(f"Maximum modelled wind: {model_max:.2f} m/s")

Maximum modelled wind: 69.24 m/s


---

## Hazard stage conclusion

**What was built.** 146 IBTrACS North Indian Ocean storms were screened to 54 entering the
study domain, standardised to a 1-hour timestep, and expanded via stochastic perturbation to
**2,754 synthetic events**. Wind fields were computed at **525 centroids** across the coastal
Odisha belt, giving a 2,754 × 525 intensity matrix.

**What was validated.**

| Check | Result |
|---|---|
| Maximum modelled wind | 69.24 m/s, consistent with severe tropical cyclone intensities in this basin |
| Sparsity | 92.9% of event–centroid pairs zero, as expected |
| Footprint structure | Coherent cyclone form: compact high-wind core with outward decay |
| Fani (2019) - location | Modelled maximum within ~25–30 km of observed maximum position |
| Fani (2019) - magnitude | 68.05 m/s modelled vs 150 kt reported; treated as a plausibility check only, since averaging conventions differ |
| Phailin (2013) | Modelled maximum ~11–12% below IBTrACS track maximum |
| Frequency preservation | Total annual event frequency maintained across the expansion |
| Tail stability | 1-in-100 wind at Puri: 50.68 m/s full catalogue, vs 41.97 and 47.14 m/s on random halves |

**Known limitation carried forward.** The split-half test shows the 1-in-100 estimate remains
sensitive to stochastic sampling. Tail estimates are therefore reported as indicative rather
than precise, and this caveat propagates to all return-period losses in `03_impact.ipynb`.

**Output:** `hazard_odisha_50.pkl` → consumed by `03_impact.ipynb`.